## checking which which unit is wrong

In [2]:
import requests
from bs4 import BeautifulSoup
import re

# Test with one property URL
test_url = "https://www.zameen.com/Property/gulberg_gulberg_3_pent_house_apartment_semi_furnished_4_beds_attched_bath_pool_gym_meeting_room_luxury_space_for_leaving-49777413-3824-1.html"  
# Replace with an actual URL from your dataset

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
})

def is_area_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) == "Area"

def clean(s):
    return re.sub(r"\s+", " ", (s or "")).strip()

r = session.get(test_url, timeout=25)
soup = BeautifulSoup(r.content, "html.parser")

area_li = soup.find(is_area_li)

if area_li:
    area_value_span = area_li.find('span', class_='_2fdf7fc5')
    if area_value_span:
        area_text = clean(area_value_span.get_text(strip=True))
        print(f"Raw area text: '{area_text}'")
        
        # Extract number and unit
        m = re.search(r"([\d.]+)\s*(Marla|Kanal|marla|kanal)?", area_text, re.IGNORECASE)
        if m:
            value = float(m.group(1))
            unit = m.group(2) if m.group(2) else "Unknown"
            print(f"Value: {value}, Unit: {unit}")

Raw area text: '1.6 Kanal'
Value: 1.6, Unit: Kanal


In [4]:
import pandas as pd
df=pd.read_csv("lahore_flats_v2.csv")
df.columns

Index(['Property ID', 'Name', 'Price', 'Marla', 'Area (sqft)', 'Floor Number',
       'Total Floors', 'Built Year', 'Address', 'Description', 'Features',
       'Nearby Locations and Other Facilities', 'Rooms', 'Other Rooms', 'Link',
       'Parking Spaces', 'Lobby in Building', 'Double Glazed Windows',
       'Central Air Conditioning', 'Central Heating', 'Flooring',
       'Electricity Backup', 'Waste Disposal', 'Floor', 'Floors in Building',
       'Elevators', 'Service Elevators in Building', 'Other Main Features',
       'Broadband Internet Access', 'Satellite or Cable TV Ready',
       'Business Center or Media Room in Building',
       'Conference Room in Building', 'Intercom', 'ATM Machines',
       'Other Business and Communication Facilities',
       'Community Lawn or Garden', 'Community Swimming Pool', 'Community Gym',
       'First Aid or Medical Centre', 'Day Care Centre', 'Kids Play Area',
       'Barbeque Area', 'Mosque', 'Community Centre',
       'Other Community Faci

In [6]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

df = pd.read_csv("lahore_flats_v2.csv")

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
})

def is_area_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) == "Area"

def clean(s):
    return re.sub(r"\s+", " ", (s or "")).strip()

def get_area_unit(url):
    # Handle null/missing URL
    if pd.isna(url) or not url:
        return "No URL"

    try:
        r = session.get(url, timeout=25)

        if r.status_code == 404:
            return "URL Expired"
        if r.status_code != 200:
            return "HTTP Error"

        soup = BeautifulSoup(r.content, "html.parser")
        area_li = soup.find(is_area_li)
        if area_li:
            area_value_span = area_li.find('span', class_='_2fdf7fc5')
            if area_value_span:
                area_text = clean(area_value_span.get_text(strip=True))
                m = re.search(r"([\d.]+)\s*(Marla|Kanal|marla|kanal)?", area_text, re.IGNORECASE)
                if m:
                    unit = m.group(2) if m.group(2) else "Unknown"
                    return unit.capitalize()
        return "Not Found"

    except requests.exceptions.Timeout:
        return "Timeout"
    except requests.exceptions.RequestException:
        return "Request Error"
    except Exception as e:
        print(f"  ⚠️ Error on {url}: {e}")
        return "Error"

# Apply to all rows
area_units = []

for idx, row in df.iterrows():
    url = row['Link']
    unit = get_area_unit(url)
    area_units.append(unit)

    # Safe print even if url is null
    url_display = url.split('/')[-1][:50] if isinstance(url, str) else "NO URL"
    print(f"[{idx+1}/{len(df)}] {url_display} → {unit}")

    time.sleep(0.5)

df['Area Unit'] = area_units

df.to_csv("lahore_flats_v2_with_unit.csv", index=False, encoding="utf-8-sig")
print(f"\n✅ Done! 'Area Unit' column added and saved.")
print(df[['Marla', 'Area Unit', 'Link']].head(10))

[1/2510] bahria_town_sector_e_bahria_town_-_johar_block_own → HTTP Error
[2/2510] bahria_town_bahria_town_-_sector_b_with_25_get_you → Marla
[3/2510] bahria_town_sector_e_bahria_town_-_johar_block_lux → Marla
[4/2510] bahria_town_bahria_town_-_sector_b_book_one_bed_lu → Marla
[5/2510] bahria_town_bahria_town_-_sector_b_with_10_discoun → Marla
[6/2510] bahria_town_bahria_town_-_sector_d_2_bed_1400_sq_f → Marla
[7/2510] bahria_town_bahria_town_-_sector_e_best_options_fo → HTTP Error
[8/2510] bahria_town_sector_e_bahria_town_-_johar_block_stu → HTTP Error
[9/2510] bahria_town_bahria_town_-_sector_e_brand_new_1-bed → Marla
[10/2510] bahria_town_bahria_town_-_sector_c_studio_flat_for → HTTP Error
[11/2510] bahria_town_sector_e_bahria_town_-_jinnah_block_bo → Marla
[12/2510] bahria_town_midway_commercial_luxury_redefined_stu → HTTP Error
[13/2510] bahria_town_sector_e_bahria_town_-_johar_block_lux → Marla
[14/2510] bahria_town_sector_e_bahria_town_-_johar_block_lux → HTTP Error
[15/2510] bah

In [8]:
df.iloc[1305][]

SyntaxError: invalid syntax (119575220.py, line 1)